# 🌸 Data Science Internship: Task 1
## Iris Flower Classification with Machine Learning

**Intern Name**: Sanket Nagathan   
**Task**: Classify Iris flowers into *Setosa*, *Versicolor*, and *Virginica* based on physical measurements.

### 1. Import Essential Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

sns.set_theme(style='whitegrid')

### 2. Ingest and Inspect the Dataset

In [ ]:
df = pd.read_csv('../data/iris.csv')
if 'Id' in df.columns:
    df = df.drop(columns=['Id'])

print(f'Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

In [ ]:
print('Missing Values Check:')
print(df.isnull().sum())

print('\nClass Distribution:')
print(df['Species'].value_counts())

df.describe()

### 3. Exploratory Data Analysis (EDA)
Visualizing relationships and distinguishing species clusters.

In [ ]:
# Pairplot showing multidimensional separability
sns.pairplot(df, hue='Species', markers=['o', 's', 'D'])
plt.suptitle('Iris Flower Feature Pairplot', y=1.02, fontsize=14, weight='bold')
plt.show()

In [ ]:
# Correlation Matrix
numeric_cols = [c for c in df.columns if c != 'Species']
plt.figure(figsize=(7, 5))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Heatmap', weight='bold')
plt.show()

### 4. Data Preprocessing & Stratified Train/Test Split

In [ ]:
X = df[numeric_cols]
encoder = LabelEncoder()
y = encoder.fit_transform(df['Species'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training Set: {X_train.shape[0]} samples')
print(f'Test Set: {X_test.shape[0]} samples')

### 5. Multi-Model Benchmarking
Comparing Logistic Regression, K-Nearest Neighbors, and Random Forest.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=300, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    print(f'=== {name} ===')
    print(f'Test Accuracy: {acc * 100:.2f}%\n')
    print(classification_report(y_test, y_pred, target_names=encoder.classes_))
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=encoder.classes_, yticklabels=encoder.classes_)
    plt.title(f'Confusion Matrix: {name}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

### 6. Interactive Sample Prediction

In [ ]:
sample = np.array([[5.1, 3.5, 1.4, 0.2]]) # Setosa measurements
sample_scaled = scaler.transform(sample)
best_clf = models['Logistic Regression']
pred = best_clf.predict(sample_scaled)[0]
probs = best_clf.predict_proba(sample_scaled)[0]

print(f'Predicted Class: {encoder.inverse_transform([pred])[0]}')
for cls, p in zip(encoder.classes_, probs):
    print(f'{cls}: {p*100:.2f}%')